In [1]:
FONTES_CET = ["santos_cet"]
CAMPOS_CET = [
    "bairro",
    "canal",
    "cnpj",
    "cpf",
    "data_de_inicio",
    "data_de_termino",
    "descricao_da_solicitacao",
    "encaminhamento_da_analise",
    "horario_de_inicio",
    "horario_de_termino",
    "nome",
    "nome_do_logradouro",
    "numero",
    "observacoes",
    "placa_do_veiculo",
    "razao_social",
    "servico_a_ser_executado",
    "tipo_de_solicitacao",
    "tipo_logradouro",
    "zona_de_restricao_de_circulacao",
]

StatementMeta(, c433abbf-a224-40aa-859f-c7cf7fbc84b8, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.functions import col, first, max as spark_max, min as spark_min, datediff, to_date

df_sol = (
    spark.table("silver_fato_solicitacoes")
    .filter(col("fonte").isin(FONTES_CET))
)

df_pivot = (
    spark.table("silver_fato_campos")
    .filter(col("fonte").isin(FONTES_CET))
    .filter(col("campo").isin(CAMPOS_CET))
    .groupBy("id_os")
    .pivot("campo", CAMPOS_CET)
    .agg(first("valor"))
)

# última etapa de cada OS
df_etapas = (
    spark.table("silver_fato_etapas")
    .filter(col("fonte").isin(FONTES_CET))
    .groupBy("id_os")
    .agg(
        spark_max("etapa").alias("etapa_atual"),
        spark_max("data_fim_etapa").alias("data_fim_ultima_etapa"),
    )
)

df_gold = (
    df_sol
    .join(df_pivot, "id_os", "left")
    .join(df_etapas, "id_os", "left")
)
# df_gold.write.mode("overwrite").format("delta").saveAsTable("gold_fato_solicitacoes_cet")

StatementMeta(, c433abbf-a224-40aa-859f-c7cf7fbc84b8, 4, Finished, Available, Finished, False)

In [3]:
display(df_gold.limit(5))

StatementMeta(, c433abbf-a224-40aa-859f-c7cf7fbc84b8, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c8683a11-8ad5-4e20-903e-dbbcb5d899cb)